# Dual-SR830 frequency and current–voltage sweep browser

Read-only analysis for standalone frequency, excitation, frequency×excitation, and temperature×excitation records. Standalone scans render separate Vxx/Vxy h1/h2/h3 figures. Temperature×excitation records render separate amplitude and phase figures for each available role and harmonic, with one curve per actual formal-window temperature. Missing data is labeled explicitly; no values are inferred.

In [ ]:
import html
import json
import math
import sys
from dataclasses import asdict
from pathlib import Path

working_directory = Path.cwd().resolve()
PROJECT_ROOT = (
    working_directory.parent
    if working_directory.name.lower() == 'notebooks'
    else working_directory
)
SOURCE_DIRECTORY = PROJECT_ROOT / 'src'
if not (SOURCE_DIRECTORY / 'attodry_control').is_dir():
    raise RuntimeError(
        f'Cannot find the project source directory: {SOURCE_DIRECTORY}'
    )
source_directory_text = str(SOURCE_DIRECTORY)
if source_directory_text not in sys.path:
    sys.path.insert(0, source_directory_text)

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from attodry_control.commissioning_analysis import (
    ExcitationPathResistance,
    HarmonicScalingRules,
    aggregate_sweep_repeatability,
    discover_commissioning_records,
    excitation_path_from_sweep_files,
    export_commissioning_csv,
    load_sweep_sample_files,
    fit_harmonic_scalings,
    plot_harmonic_scaling_fit,
    plot_multi_frequency_iv_curves,
    plot_six_role_harmonic_sweeps,
    plot_sweep_repeatability,
)
from attodry_control.temperature_excitation_analysis import (
    discover_temperature_excitation_records,
    export_temperature_excitation_csv,
    load_temperature_excitation_sample_files,
    plot_temperature_iv_suite,
)
from attodry_control.scientific_plotting import export_publication_figure_set

DATA_DIRECTORY = PROJECT_ROOT / 'run_data' / 'commissioning'
TEMPERATURE_DATA_DIRECTORY = (
    PROJECT_ROOT / 'run_data' / 'temperature_excitation_commissioning'
)

# Analysis-only thresholds. Edit these values for the current data-quality
# decision; they never change the hardware sweep or safety protocol.
SCALING_RULES = HarmonicScalingRules(
    confidence_level=0.95,
    minimum_points=6,
    minimum_current_decades=1.0,
    minimum_snr=3.0,
    max_exponent_ci_width=0.5,
    max_delta_aicc_consistent=2.0,
    min_delta_aicc_inconsistent=6.0,
    max_relative_rmse=0.10,
    max_phase_slope_deg_per_decade=5.0,
    max_phase_span_deg=10.0,
    # Scalar R background: 'auto', 'none', or 'with_offset'.
    scalar_background_mode='auto',
    # Complex Z background: 'auto', 'none', or 'with_offset'.
    complex_background_mode='auto',
    complex_free_exponent_min=0.05,
    complex_free_exponent_max=6.0,
)

# Choose which already-computed harmonic-scaling fits are drawn. Start with
# all three methods for comparison; this setting does not rerun or alter fits.
SCALING_PLOT_METHODS = ("log", "scalar", "complex")
# After comparing the methods, replace the line above with ONE example below:
# SCALING_PLOT_METHODS = ("scalar",)   # linear-coordinate R fit; p is still fitted
# SCALING_PLOT_METHODS = ("log",)      # log(R) versus log(I) fit only
# SCALING_PLOT_METHODS = ("complex",)  # phase-preserving Z=X+iY fit only
# Keep the trailing comma: ("scalar",) is a one-item tuple.

# Configure data directories above; plotting cells display figures only.

## Temperature-stacked current–Vxx/Vxy and phase curves

This section reads completed temperature conditions from temperature–excitation summary JSON or formal CSV files. Set `TEMPERATURE_DATA_DIRECTORY` in the first cell, refresh the catalog, select one or more records, then load them. Select temperature conditions and an optional archived RMS-current interval; figures use only their intersection. The current coordinate is the recorded readback-derived `nominal_current_a_rms`; it is never recalculated from today's TOML. Each available Vxx/Vxy × h1/h2/h3 channel produces one amplitude figure and one phase figure, with one colored curve per selected condition. Legend temperatures are actual formal-window means, not requested setpoints, and are drawn outside the plot frame on the right.

Phase repeats use circular mean and circular standard deviation, then each temperature curve is unwrapped only along increasing current for display. No phase setting or raw value is changed. The default sample filter is `clean`; selecting another status is an explicit audit view.

In [ ]:
temperature_sample_status_widget = widgets.SelectMultiple(
    options=('clean', 'problem', 'unlocked', 'overload', 'instrument_error'),
    value=('clean',),
    description='T formal status',
    layout=widgets.Layout(width='95%', height='100px'),
)
temperature_excitation_record_widget = widgets.SelectMultiple(
    options=(),
    description='T × excitation',
    layout=widgets.Layout(width='95%', height='150px'),
)
temperature_condition_widget = widgets.SelectMultiple(
    options=(),
    description='Temperatures',
    layout=widgets.Layout(width='95%', height='170px'),
)
temperature_current_minimum_widget = widgets.Text(
    value='',
    placeholder='blank = no lower limit',
    description='I min (A RMS)',
    layout=widgets.Layout(width='47%'),
)
temperature_current_maximum_widget = widgets.Text(
    value='',
    placeholder='blank = no upper limit',
    description='I max (A RMS)',
    layout=widgets.Layout(width='47%'),
)
refresh_temperature_records_button = widgets.Button(
    description='Refresh T records', icon='refresh', button_style='info'
)
load_temperature_records_button = widgets.Button(
    description='Load T records', icon='check', button_style='success'
)
apply_temperature_filter_button = widgets.Button(
    description='Apply T × I filters', icon='filter', button_style='warning'
)
temperature_selector_message = widgets.HTML()

TEMPERATURE_EXCITATION_PATHS = ()
temperature_excitation_paths = ()
temperature_excitation_loaded_rows = ()
temperature_excitation_rows = ()
temperature_current_minimum_a_rms = None
temperature_current_maximum_a_rms = None
temperature_iv_figures = {}

def _temperature_condition_key(row):
    return f'{row.source_path}::{row.temperature_index}'

def _temperature_condition_options(rows):
    grouped = {}
    for row in rows:
        grouped.setdefault(_temperature_condition_key(row), row)
    return tuple(
        (
            f'{row.condition_measurement_temperature_k:.6g} K measured | '
            f'{row.requested_temperature_k:.6g} K set | '
            f'{Path(row.source_path).name} | T#{row.temperature_index}',
            key,
        )
        for key, row in sorted(
            grouped.items(),
            key=lambda item: (
                item[1].condition_measurement_temperature_k,
                item[1].source_path,
                item[1].temperature_index,
            ),
        )
    )

def _parse_optional_current_bound(value, label):
    text = value.strip()
    if not text:
        return None
    try:
        current = float(text)
    except ValueError as exc:
        raise ValueError(f'{label} must be a number in A RMS or blank.') from exc
    if not math.isfinite(current) or current < 0.0:
        raise ValueError(f'{label} must be a finite non-negative current.')
    return current

def _refresh_temperature_records(_=None):
    records = discover_temperature_excitation_records(
        TEMPERATURE_DATA_DIRECTORY
    )
    temperature_excitation_record_widget.options = tuple(
        (path.name, str(path)) for path in records
    )
    temperature_selector_message.value = (
        f'<b>Directory:</b> {TEMPERATURE_DATA_DIRECTORY}<br>'
        f'Found {len(records)} temperature–excitation records.'
    )

def _load_temperature_records(_=None):
    global TEMPERATURE_EXCITATION_PATHS
    global temperature_excitation_paths, temperature_excitation_loaded_rows
    global temperature_excitation_rows
    TEMPERATURE_EXCITATION_PATHS = tuple(
        Path(value) for value in temperature_excitation_record_widget.value
    )
    if not TEMPERATURE_EXCITATION_PATHS:
        temperature_selector_message.value = '<b>No temperature–excitation record selected.</b>'
        return
    try:
        temperature_excitation_loaded_rows = load_temperature_excitation_sample_files(
            TEMPERATURE_EXCITATION_PATHS,
            sample_statuses=set(temperature_sample_status_widget.value),
        )
    except ValueError as exc:
        temperature_selector_message.value = f'<b>Could not load temperature data:</b> {exc}'
        return
    temperature_excitation_paths = TEMPERATURE_EXCITATION_PATHS
    temperature_condition_widget.options = _temperature_condition_options(
        temperature_excitation_loaded_rows
    )
    temperature_condition_widget.value = tuple(
        value for _label, value in temperature_condition_widget.options
    )
    temperature_excitation_rows = temperature_excitation_loaded_rows
    temperature_selector_message.value = (
        f'<b>Loaded:</b> {len(temperature_excitation_paths)} file(s), '
        f'{len(temperature_excitation_rows)} selected formal rows, '
        f'{len(temperature_condition_widget.options)} temperature conditions.'
    )

def _apply_temperature_filter(_=None):
    global temperature_excitation_rows
    global temperature_current_minimum_a_rms
    global temperature_current_maximum_a_rms
    selected = set(temperature_condition_widget.value)
    try:
        minimum_current = _parse_optional_current_bound(
            temperature_current_minimum_widget.value, 'I min'
        )
        maximum_current = _parse_optional_current_bound(
            temperature_current_maximum_widget.value, 'I max'
        )
    except ValueError as exc:
        temperature_selector_message.value = f'<b>Current filter error:</b> {exc}'
        return
    if (
        minimum_current is not None
        and maximum_current is not None
        and minimum_current > maximum_current
    ):
        temperature_selector_message.value = '<b>Current filter error:</b> I min cannot exceed I max.'
        return
    temperature_excitation_rows = tuple(
        row
        for row in temperature_excitation_loaded_rows
        if (
            _temperature_condition_key(row) in selected
            and (minimum_current is None or row.current_a_rms >= minimum_current)
            and (maximum_current is None or row.current_a_rms <= maximum_current)
        )
    )
    temperature_current_minimum_a_rms = minimum_current
    temperature_current_maximum_a_rms = maximum_current
    minimum_label = 'no lower limit' if minimum_current is None else f'{minimum_current:.6g}'
    maximum_label = 'no upper limit' if maximum_current is None else f'{maximum_current:.6g}'
    temperature_selector_message.value = (
        f'<b>T × I selection applied:</b> {len(selected)} condition(s), '
        f'I = {minimum_label} to {maximum_label} A RMS, '
        f'{len(temperature_excitation_rows)} formal rows. Run the figure cell.'
    )

refresh_temperature_records_button.on_click(_refresh_temperature_records)
load_temperature_records_button.on_click(_load_temperature_records)
apply_temperature_filter_button.on_click(_apply_temperature_filter)
_refresh_temperature_records()
display(widgets.VBox([
    widgets.HBox([refresh_temperature_records_button, load_temperature_records_button]),
    temperature_sample_status_widget,
    temperature_excitation_record_widget,
    temperature_condition_widget,
    widgets.HBox([
        temperature_current_minimum_widget,
        temperature_current_maximum_widget,
    ]),
    apply_temperature_filter_button,
    temperature_selector_message,
]))

In [ ]:
temperature_iv_figures = (
    plot_temperature_iv_suite(temperature_excitation_rows)
    if temperature_excitation_rows
    else {}
)
for figure in temperature_iv_figures.values():
    display(figure)
    plt.close(figure)

## Start here: filters, remote record selection, and current calibration

Set `DATA_DIRECTORY` once in the preceding cell. Click **Refresh records**, tick the boxes to the right of any records in **Frequency**, **Excitation**, and **f × e**, then click **Load selected records**. Each entry shows its number of formal samples and the XX/XY harmonics actually recorded. Each category accepts one or multiple files independently; refreshing preserves checked files still in the catalog. A frequency record produces frequency figures; an excitation record produces current-voltage figures; f × e produces frequency-indexed current-voltage curves. The `completed` checkbox is on by default; deselect it only for an explicit audit. `clean` formal samples are the default automatic quality screen. Current is always `SINE OUT RMS voltage / (external series + SR830 output + approximate device resistance)`, using the path archived with each sweep from `hardware.local.toml`. Do not duplicate normal resistance values here.

### Harmonic scaling decision rules

The `SCALING_RULES` block in the first code cell is intentionally editable. It is analysis-only and is copied into `selection_manifest.json` when outputs are saved. `SCALING_PLOT_METHODS` controls only which fitted curves, equations, verdicts, and residuals are drawn; all three methods are still calculated and exported for audit. Start with `("log", "scalar", "complex")` to compare them, then use a one-item tuple such as `("scalar",)` after choosing a method. For each available XX/XY and h1/h2/h3 excitation channel, the analysis compares three distinct views. The log-magnitude fit is `log R = log A + p log I` against fixed `p=n`. The phase-blind scalar fit is `R = b + A·(I/Iref)^p` against fixed `p=n`; it uses only measured amplitude R, ignores phase and X/Y, and constrains b and A to be non-negative. Selecting `scalar` means fitting in linear R coordinates; it does not force `p=1`. `scalar_phase_ignored=True` is retained in every exported result so this choice is explicit. The complex fit is performed directly on X/Y.

`minimum_points` is the minimum number of current points; `minimum_current_decades` is the required log10(Imax/Imin) span; `minimum_snr` excludes a point only when replicate standard error is available and the estimated SNR is below the threshold. `max_exponent_ci_width` limits the width of the approximate confidence interval for the fitted exponent. `max_delta_aicc_consistent` and `min_delta_aicc_inconsistent` compare fixed-order and free-exponent models, where ΔAICc = AICc_fixed − AICc_free. `max_relative_rmse` limits the fixed-order relative error. Set optional thresholds to `None` to disable that criterion.

For the scalar fit, `scalar_background_mode='auto'` uses corrected AIC to choose between no-offset and offset fixed-order models; `'none'` or `'with_offset'` forces one. The four scalar models and `scalar_R_verdict` are exported. Their AICc values are comparable within the scalar method, but should not be compared directly with log-space or complex-space AICc because those methods use different residual spaces.
The complex, background-aware comparison works on `Z=X+iY`, not on magnitude alone. It calculates four models: `Z=C·I^n`, `Z=C·I^p`, `Z=B+C·I^n`, and `Z=B+C·I^p`. Here `B` is a complex background with its own amplitude and phase. `complex_background_mode='auto'` uses corrected AIC to choose whether the fixed-order comparison uses `B`; `'none'` forces no background and `'with_offset'` forces it. `complex_power_law_verdict` is the offset-aware conclusion; its `complex_models` export contains the fitted `B`, response vector at the geometric-mean current reference, AICc, residual, exponent, and confidence interval. Raw phase can rotate because `B` and `C·I^n` have different phases, so `complex_response_verdict` remains a separate raw-phase quality check. `R²` is contextual only and never the sole decision rule.

In [ ]:
def _labeled_control(title, control):
    return widgets.VBox(
        [widgets.HTML(value=f'<b>{html.escape(title)}</b>'), control],
        layout=widgets.Layout(width='100%', min_width='0'),
    )

class _RecordCheckboxList:
    """Full-width records with independent checkboxes on the right."""
    def __init__(self, title):
        self.title = title
        self._options = ()
        self._checkboxes = {}
        self._rows = widgets.VBox(
            layout=widgets.Layout(width='100%', min_width='0', max_height='240px', overflow='auto')
        )
        self.widget = self._rows
        self.options = ()

    @property
    def options(self):
        return self._options

    @options.setter
    def options(self, options):
        selected = set(self.value)
        self._options = tuple(options)
        self._checkboxes = {}
        rows = []
        for label, path in self._options:
            checkbox = widgets.Checkbox(
                value=path in selected, description='', indent=False,
                tooltip=f'Select {Path(path).name}',
                layout=widgets.Layout(width='28px', flex='0 0 28px'),
            )
            self._checkboxes[path] = checkbox
            record_label = widgets.HTML(
                value=f'<div style="white-space:normal;overflow-wrap:anywhere">{html.escape(label)}</div>',
                layout=widgets.Layout(width='auto', min_width='0', flex='1 1 0%'),
            )
            rows.append(widgets.HBox(
                [record_label, checkbox],
                layout=widgets.Layout(width='100%', min_width='0', align_items='center'),
            ))
        self._rows.children = tuple(rows) or (widgets.HTML(value='No records found.'),)

    @property
    def value(self):
        return tuple(path for path, checkbox in self._checkboxes.items() if checkbox.value)

    @value.setter
    def value(self, paths):
        selected = set(paths)
        if not selected.issubset(self._checkboxes):
            raise ValueError('Selected record is not in the current catalog.')
        for path, checkbox in self._checkboxes.items():
            checkbox.value = path in selected

completed_only_widget = widgets.Checkbox(
    value=True,
    description='Only completed records',
    indent=False, layout=widgets.Layout(width='auto', min_width='210px'),
)
sample_status_widget = widgets.SelectMultiple(
    options=('clean', 'problem', 'unlocked', 'overload', 'instrument_error'),
    value=('clean',),
    description='',
    layout=widgets.Layout(width='100%'),
)
include_rejected_widget = widgets.Checkbox(
    value=False,
    description='Allow rejected audit records',
    indent=False, layout=widgets.Layout(width='auto', min_width='230px'),
)
refresh_records_button = widgets.Button(
    description='Refresh records',
    icon='refresh',
    button_style='info',
)
frequency_record_widget = _RecordCheckboxList('Frequency')
excitation_record_widget = _RecordCheckboxList('Excitation')
combined_record_widget = _RecordCheckboxList('f × e (frequency × excitation)')
frequency_baseline_widget = widgets.Dropdown(
    options=(), description='', layout=widgets.Layout(width='100%'),
)
excitation_baseline_widget = widgets.Dropdown(
    options=(), description='', layout=widgets.Layout(width='100%'),
)
combined_baseline_widget = widgets.Dropdown(
    options=(), description='', layout=widgets.Layout(width='100%'),
)
repeatability_metrics_widget = widgets.SelectMultiple(
    options=(('R amplitude', 'amplitude_v'), ('X', 'x_v'), ('Y', 'y_v'), ('Phase', 'phase_deg')),
    value=('amplitude_v',), description='',
    layout=widgets.Layout(width='100%', height='100px'),
)
load_selected_records_button = widgets.Button(
    description='Load selected records',
    layout=widgets.Layout(width='auto', min_width='200px'),
    icon='check',
    button_style='success',
)
apply_point_exclusions_button = widgets.Button(
    description='Apply point exclusions',
    layout=widgets.Layout(width='auto', min_width='200px'),
    icon='filter',
    button_style='warning',
)
frequency_excluded_points_widget = widgets.SelectMultiple(
    options=(),
    description='',
    layout=widgets.Layout(width='100%', height='120px'),
)
excitation_excluded_points_widget = widgets.SelectMultiple(
    options=(),
    description='',
    layout=widgets.Layout(width='100%', height='120px'),
)
combined_excluded_frequencies_widget = widgets.SelectMultiple(
    options=(), description='', layout=widgets.Layout(width='100%', height='120px'),
)
combined_excluded_excitations_widget = widgets.SelectMultiple(
    options=(), description='', layout=widgets.Layout(width='100%', height='120px'),
)
selector_message = widgets.HTML(layout=widgets.Layout(width='100%', min_width='0'))
point_filter_message = widgets.HTML(layout=widgets.Layout(width='100%', min_width='0'))
excitation_x_axis_widget = widgets.Dropdown(
    options=(('Calculated current (A RMS)', 'sine_output_current_a_rms'),
             ('SINE OUT readback (V RMS)', 'sine_output_v_rms')),
    value='sine_output_current_a_rms', description='Excitation X:',
)

FREQUENCY_PATHS = ()
EXCITATION_PATHS = ()
COMBINED_PATHS = ()
FREQUENCY_EXCLUDED_TARGET_HZ = set()
EXCITATION_EXCLUDED_SOURCE_V_RMS = set()
COMBINED_EXCLUDED_FREQUENCIES_HZ = set()
COMBINED_EXCLUDED_EXCITATIONS_V_RMS = set()

def _sync_filters():
    global RECORD_STATUSES, SAMPLE_STATUSES, INCLUDE_REJECTED
    RECORD_STATUSES = {'completed'} if completed_only_widget.value else None
    SAMPLE_STATUSES = set(sample_status_widget.value)
    INCLUDE_REJECTED = include_rejected_widget.value

def _record_options(scan_type):
    records = discover_commissioning_records(
        DATA_DIRECTORY,
        record_statuses=RECORD_STATUSES,
        scan_types={scan_type},
    )
    return [
        (
            _record_label(record),
            str(record.path),
        )
        for record in records
    ]

def _record_label(record):
    by_role = {}
    for role, harmonic in record.channels:
        by_role.setdefault(role, []).append(harmonic)
    channel_labels = [
        f"{role.upper()} " + '/'.join(f'h{harmonic}' for harmonic in harmonics)
        for role, harmonics in by_role.items()
    ]
    channels = ', '.join(channel_labels) or 'no recorded channels'
    return f'{record.path.name} | {record.sample_count} formal samples | {channels}'

def _set_baseline_options(widget, paths, rows):
    available = {row.source_path for row in rows}
    options = tuple((Path(path).name, str(path)) for path in paths if str(path) in available)
    previous = widget.value
    widget.value = None
    widget.options = options
    valid = {value for _, value in options}
    widget.value = previous if previous in valid else (options[0][1] if options else None)


def _refresh_records(_=None):
    _sync_filters()
    frequency_options = _record_options('frequency')
    excitation_options = _record_options('excitation')
    combined_options = _record_options('frequency_excitation')
    frequency_record_widget.options = frequency_options
    excitation_record_widget.options = excitation_options
    combined_record_widget.options = combined_options
    selector_message.value = (
        f'<b>Directory:</b> {DATA_DIRECTORY}<br>'
        f'Found {len(frequency_options)} frequency, {len(excitation_options)} excitation, '
        f'and {len(combined_options)} frequency×amplitude records.'
    )

def _load_selected_records(_):
    global FREQUENCY_PATHS, EXCITATION_PATHS, COMBINED_PATHS
    global frequency_rows, excitation_rows, combined_rows
    global frequency_loaded_rows, excitation_loaded_rows, combined_loaded_rows
    _sync_filters()
    FREQUENCY_PATHS = tuple(Path(path) for path in frequency_record_widget.value)
    EXCITATION_PATHS = tuple(Path(path) for path in excitation_record_widget.value)
    COMBINED_PATHS = tuple(Path(path) for path in combined_record_widget.value)
    frequency_rows = excitation_rows = combined_rows = ()
    frequency_loaded_rows = excitation_loaded_rows = combined_loaded_rows = ()
    selected_types = []
    if FREQUENCY_PATHS:
        selected_types.append(f"frequency: {len(FREQUENCY_PATHS)} run(s)")
    if EXCITATION_PATHS:
        selected_types.append(f"excitation: {len(EXCITATION_PATHS)} run(s)")
    if COMBINED_PATHS:
        selected_types.append(f"frequency×amplitude: {len(COMBINED_PATHS)} run(s)")
    if not selected_types:
        _load_selected_formal_samples()
        selector_message.value = (
            '<b>No record selected.</b> Choose at least one frequency, excitation, '
            'or frequency×amplitude record, then load it.'
        )
        frequency_baseline_widget.options = ()
        excitation_baseline_widget.options = ()
        combined_baseline_widget.options = ()
        return
    try:
        loaded = _load_selected_formal_samples()
    except ValueError as exc:
        selector_message.value = f'<b>Could not load selected samples:</b> {exc}'
        return
    selector_message.value = (
        '<b>Loaded:</b> ' + '; '.join(selected_types) + '<br>'
        f"Points ready: {loaded['frequency_selected_rows']} frequency rows; "
        f"{loaded['excitation_selected_rows']} excitation rows; "
        f"{loaded['combined_selected_rows']} combined rows. Choose any "
        'points to exclude, then run the figure cell.'
    )

def _coordinate_exclusion_options(rows, coordinate, label):
    point_indices_by_value = {}
    for row in rows:
        value = coordinate(row)
        if value is not None and math.isfinite(value):
            point_indices_by_value.setdefault(value, set()).add(row.point_index)
    return tuple(
        (
            f'{label(value)} | point(s) ' + ', '.join(
                f'#{index}' for index in sorted(point_indices)
            ),
            value,
        )
        for value, point_indices in sorted(point_indices_by_value.items())
    )

def _load_selected_formal_samples():
    _sync_filters()
    global FREQUENCY_EXCLUDED_TARGET_HZ, EXCITATION_EXCLUDED_SOURCE_V_RMS
    global frequency_paths, excitation_paths, combined_paths
    global frequency_loaded_rows, excitation_loaded_rows, combined_loaded_rows
    global COMBINED_EXCLUDED_FREQUENCIES_HZ, COMBINED_EXCLUDED_EXCITATIONS_V_RMS
    global frequency_excitation_path, excitation_excitation_path, combined_excitation_path
    global frequency_rows, excitation_rows, combined_rows
    frequency_paths = tuple(FREQUENCY_PATHS)
    excitation_paths = tuple(EXCITATION_PATHS)
    combined_paths = tuple(COMBINED_PATHS)
    frequency_loaded_rows = (
        load_sweep_sample_files(
            frequency_paths,
            include_rejected=INCLUDE_REJECTED,
            sample_statuses=SAMPLE_STATUSES,
        )
        if frequency_paths
        else ()
    )
    excitation_loaded_rows = (
        load_sweep_sample_files(
            excitation_paths,
            include_rejected=INCLUDE_REJECTED,
            sample_statuses=SAMPLE_STATUSES,
        )
        if excitation_paths
        else ()
    )
    combined_loaded_rows = (
        load_sweep_sample_files(
            combined_paths,
            include_rejected=INCLUDE_REJECTED,
            sample_statuses=SAMPLE_STATUSES,
        )
        if combined_paths
        else ()
    )
    frequency_excitation_path = (
        excitation_path_from_sweep_files(
            frequency_paths,
            excitation_path_override=EXCITATION_PATH_OVERRIDE,
        )
        if frequency_paths
        else None
    )
    excitation_excitation_path = (
        excitation_path_from_sweep_files(
            excitation_paths,
            excitation_path_override=EXCITATION_PATH_OVERRIDE,
        )
        if excitation_paths and excitation_x_axis_widget.value == 'sine_output_current_a_rms'
        else None
    )
    combined_excitation_path = (
        excitation_path_from_sweep_files(
            combined_paths,
            excitation_path_override=EXCITATION_PATH_OVERRIDE,
        )
        if combined_paths and excitation_x_axis_widget.value == 'sine_output_current_a_rms'
        else None
    )
    _set_baseline_options(frequency_baseline_widget, frequency_paths, frequency_loaded_rows)
    _set_baseline_options(excitation_baseline_widget, excitation_paths, excitation_loaded_rows)
    _set_baseline_options(combined_baseline_widget, combined_paths, combined_loaded_rows)
    frequency_excluded_points_widget.options = _coordinate_exclusion_options(
        frequency_loaded_rows,
        lambda row: row.target_frequency_hz,
        lambda value: f'{value:g} Hz target',
    )
    excitation_excluded_points_widget.options = _coordinate_exclusion_options(
        excitation_loaded_rows,
        lambda row: row.source_v_rms,
        lambda value: (f'{excitation_excitation_path.current_from_sine_output(value):.4g} A RMS target'
                       if excitation_excitation_path is not None else f'{value:.4g} V RMS target'),
    )
    combined_excluded_frequencies_widget.options = _coordinate_exclusion_options(
        combined_loaded_rows,
        lambda row: row.target_frequency_hz,
        lambda value: f'{value:g} Hz target',
    )
    combined_excluded_excitations_widget.options = _coordinate_exclusion_options(
        combined_loaded_rows,
        lambda row: row.source_v_rms,
        lambda value: (f'{combined_excitation_path.current_from_sine_output(value):.4g} A RMS target'
                       if combined_excitation_path is not None else f'{value:.4g} V RMS target'),
    )
    frequency_excluded_points_widget.value = ()
    excitation_excluded_points_widget.value = ()
    combined_excluded_frequencies_widget.value = ()
    combined_excluded_excitations_widget.value = ()
    FREQUENCY_EXCLUDED_TARGET_HZ = set()
    EXCITATION_EXCLUDED_SOURCE_V_RMS = set()
    COMBINED_EXCLUDED_FREQUENCIES_HZ = set()
    COMBINED_EXCLUDED_EXCITATIONS_V_RMS = set()
    frequency_rows = frequency_loaded_rows
    excitation_rows = excitation_loaded_rows
    combined_rows = combined_loaded_rows
    point_filter_message.value = (
        '<b>Automatic quality screen:</b> ' + ', '.join(sorted(SAMPLE_STATUSES)) +
        '. Exclusions apply across selected files. Unequal point counts are supported; '        'paired differences use only shared requested coordinates.'
    )
    return {
        'frequency_files': frequency_paths,
        'frequency_selected_rows': len(frequency_rows),
        'frequency_total_path_resistance_ohm': (
            frequency_excitation_path.total_resistance_ohm
            if frequency_excitation_path is not None
            else None
        ),
        'excitation_files': excitation_paths,
        'excitation_selected_rows': len(excitation_rows),
        'excitation_total_path_resistance_ohm': (
            excitation_excitation_path.total_resistance_ohm
            if excitation_excitation_path is not None
            else None
        ),
        'combined_files': combined_paths,
        'combined_selected_rows': len(combined_rows),
        'combined_total_path_resistance_ohm': (
            combined_excitation_path.total_resistance_ohm
            if combined_excitation_path is not None
            else None
        ),
    }

def _rows_after_coordinate_exclusions(rows, attribute, excluded_values):
    return tuple(row for row in rows if getattr(row, attribute) not in excluded_values)

def _apply_point_exclusions(_):
    global FREQUENCY_EXCLUDED_TARGET_HZ, EXCITATION_EXCLUDED_SOURCE_V_RMS
    global COMBINED_EXCLUDED_FREQUENCIES_HZ, COMBINED_EXCLUDED_EXCITATIONS_V_RMS
    global frequency_rows, excitation_rows, combined_rows
    FREQUENCY_EXCLUDED_TARGET_HZ = set(frequency_excluded_points_widget.value)
    EXCITATION_EXCLUDED_SOURCE_V_RMS = set(excitation_excluded_points_widget.value)
    COMBINED_EXCLUDED_FREQUENCIES_HZ = set(combined_excluded_frequencies_widget.value)
    COMBINED_EXCLUDED_EXCITATIONS_V_RMS = set(combined_excluded_excitations_widget.value)
    combined_rows = tuple(
        row for row in globals().get('combined_loaded_rows', ())
        if row.target_frequency_hz not in COMBINED_EXCLUDED_FREQUENCIES_HZ
        and row.source_v_rms not in COMBINED_EXCLUDED_EXCITATIONS_V_RMS
    )
    frequency_rows = _rows_after_coordinate_exclusions(
        globals().get('frequency_loaded_rows', ()), 'target_frequency_hz', FREQUENCY_EXCLUDED_TARGET_HZ
    )
    excitation_rows = _rows_after_coordinate_exclusions(
        globals().get('excitation_loaded_rows', ()), 'source_v_rms', EXCITATION_EXCLUDED_SOURCE_V_RMS
    )
    point_filter_message.value = (
        f'<b>Plot selection applied.</b> Frequency: {len(frequency_rows)} rows '
        f'after excluding {len(FREQUENCY_EXCLUDED_TARGET_HZ)} shared frequencies; '
        f'excitation: {len(excitation_rows)} rows after excluding {len(EXCITATION_EXCLUDED_SOURCE_V_RMS)} shared currents; '
        f'f × e: {len(combined_rows)} rows after excluding '
        f'{len(COMBINED_EXCLUDED_FREQUENCIES_HZ)} frequencies and '
        f'{len(COMBINED_EXCLUDED_EXCITATIONS_V_RMS)} excitation values. Run the figure cell.'
    )

record_categories_widget = widgets.Accordion(children=(
    frequency_record_widget.widget,
    excitation_record_widget.widget,
    combined_record_widget.widget,
))
record_categories_widget.set_title(0, 'Frequency')
record_categories_widget.set_title(1, 'Excitation')
record_categories_widget.set_title(2, 'f × e')
record_categories_widget.selected_index = None

repeatability_options_widget = widgets.Accordion(children=(widgets.VBox([
    _labeled_control('Frequency baseline', frequency_baseline_widget),
    _labeled_control('Excitation baseline', excitation_baseline_widget),
    _labeled_control('f × e baseline', combined_baseline_widget),
    _labeled_control('Compare metrics', repeatability_metrics_widget),
]),))
repeatability_options_widget.set_title(0, 'Multi-run paired comparison options')
repeatability_options_widget.selected_index = None

refresh_records_button.on_click(_refresh_records)
load_selected_records_button.on_click(_load_selected_records)
apply_point_exclusions_button.on_click(_apply_point_exclusions)
_refresh_records()
record_selector_panel = widgets.VBox([
    widgets.HBox(
        [refresh_records_button, load_selected_records_button, completed_only_widget, include_rejected_widget],
        layout=widgets.Layout(width='100%', flex_flow='row wrap'),
    ),
    _labeled_control('Formal samples', sample_status_widget),
    record_categories_widget,
    repeatability_options_widget,
    excitation_x_axis_widget,
    selector_message,
    _labeled_control('Exclude frequency points', frequency_excluded_points_widget),
    _labeled_control('Exclude excitation points', excitation_excluded_points_widget),
    _labeled_control('f × e: exclude frequencies', combined_excluded_frequencies_widget),
    _labeled_control('f × e: exclude excitations', combined_excluded_excitations_widget),
    apply_point_exclusions_button,
    point_filter_message,
], layout=widgets.Layout(width='100%', min_width='0', align_items='stretch'))
display(record_selector_panel)

_sync_filters()

# Default: use measurement_config.excitation_path recorded in each selected JSON.
# Set this only for legacy JSON that lacks that snapshot; it deliberately
# overrides every selected file, so do not use it for normal daily records.
EXCITATION_PATH_OVERRIDE: ExcitationPathResistance | None = None
# EXCITATION_PATH_OVERRIDE = ExcitationPathResistance(
#     external_series_resistance_ohm=...,
#     sr830_output_resistance_ohm=...,
#     approximate_device_resistance_ohm=...,
# )
# Plot phase only above this amplitude and below this circular sample spread.
# Set the amplitude to 0.0 and the spread to None to inspect all raw phases.
PHASE_MINIMUM_AMPLITUDE_V = 1e-6
PHASE_MAXIMUM_STANDARD_DEVIATION_DEG = 5.0
# The selection UI above sets these tuples. The notebook never silently
# substitutes a newer record when no record of that type was selected.

# Plot settings above take effect when the figure cell is run.

## Filtered catalog

The catalog is newest-first. Toggle the completed checkbox, choose formal-sample statuses, then click **Refresh records**. Tick files independently in the three categories; the notebook does not silently substitute a newer record.

In [ ]:
RECORD_STATUSES = {'completed'} if completed_only_widget.value else None
SAMPLE_STATUSES = set(sample_status_widget.value)
INCLUDE_REJECTED = include_rejected_widget.value

# The controls above refresh and show the filtered record catalog.

## Load selected formal samples

Excitation X chooses either current calculated from the archived resistance or the raw SR830 SINE OUT voltage readback for excitation and f × e curves. After switching, click Load selected records and rerun the figure cell. Raw-voltage overlays allow different archived resistance profiles; current overlays require a common path. Harmonic fits and the condensed report remain current-based, with each run's own profile.

The loader excludes transition and cleanup payloads. It refuses rejected records unless `INCLUDE_REJECTED=True`. All three record categories stay separate and any can be empty. Clicking **Load selected records** fills the exclusion lists and resets previous exclusions. Frequency and excitation keep their per-file point exclusions. For **f × e**, select unwanted requested frequencies and/or SINE OUT excitation values in the two separate lists: a frequency removes its whole row across excitations, and an excitation removes its whole column across frequencies, in every selected f × e file. Readback variations do not split these requested coordinates. Click **Apply point exclusions**, then rerun the figure cell. Clear the exclusions and apply again to restore all automatically retained points.

In [ ]:
# This remains available for rerunning after changing a formal-sample filter.
# The Load selected records button already calls it for the normal workflow.
_ = _load_selected_formal_samples()

## Available frequency and current–voltage figures

Tick multiple records of the same scan type and load them, then choose a baseline run and one or more metrics before running the figure cell. Multi-run figures overlay each run's mean ± within-run spread and add a paired-difference panel against the chosen baseline; unmatched requested coordinates are omitted rather than interpolated. Frequency runs pair by requested frequency, excitation runs by requested SINE OUT voltage, and combined runs by both requested coordinates. Curves still use measured frequency/current readbacks. The archived settings table above helps identify condition changes. Harmonic fits and condensed reports are computed separately per run, never pooled across files.

A loaded frequency record produces six frequency figures with a logarithmic frequency axis and SINE OUT-derived RMS current in their titles. A loaded excitation record produces six current-voltage figures using the same calculated current. Each figure keeps voltage magnitude and phase on separate y axes. Missing scan types are skipped; missing harmonics are labeled explicitly.

For excitation data, the next part of the cell performs the editable harmonic-scaling analysis. It fits every available XX/XY and h1/h2/h3 combination independently and plots the log-space fits, the selected phase-blind scalar-R curve, and the selected complex-background curve. The Notebook displays figures only: legends are outside the axes on the right and numerical fit results are retained in the optional export manifest rather than rendered in the Notebook. Use `scalar_R_verdict` when phase is too unstable to trust; it is explicitly an amplitude-only result. Use `complex_power_law_verdict` for the background-aware X/Y result; `complex_response_verdict` remains a separate raw-phase stability audit.

In [ ]:
frequency_figures = {}
current_voltage_figures = {}
combined_iv_figures = {}
harmonic_scaling_results_by_run = {}
harmonic_scaling_results = {}
harmonic_scaling_figures = {}
repeatability_summary = []

def _repeatability_figures(rows, baseline, excitation_path):
    figures = {}
    for role in ('xx', 'xy'):
        for harmonic in (1, 2, 3):
            if not any(row.role == role and row.harmonic == harmonic for row in rows):
                continue
            for metric in repeatability_metrics_widget.value:
                statistics = aggregate_sweep_repeatability(
                    rows, role=role, harmonic=harmonic, metric=metric,
                    excitation_path=excitation_path,
                    excitation_x_axis=excitation_x_axis_widget.value,
                )
                by_run = {}
                for statistic in statistics:
                    by_run.setdefault(statistic.source_path, []).append(statistic)
                baseline_points = {
                    statistic.coordinates: statistic
                    for statistic in by_run.get(baseline, ())
                }
                for source_path, run_points in by_run.items():
                    if source_path == baseline:
                        continue
                    deltas = []
                    for statistic in run_points:
                        reference = baseline_points.get(statistic.coordinates)
                        if reference is None:
                            continue
                        delta = (
                            (statistic.mean - reference.mean + 180.0) % 360.0 - 180.0
                            if metric == 'phase_deg'
                            else statistic.mean - reference.mean
                        )
                        deltas.append(delta)
                    absolute_deltas = [abs(delta) for delta in deltas]
                    repeatability_summary.append({
                        'scan_type': rows[0].scan_type,
                        'source_path': source_path,
                        'baseline_source_path': baseline,
                        'role': role,
                        'harmonic': harmonic,
                        'metric': metric,
                        'matched_coordinates': len(deltas),
                        'mean_absolute_difference': (
                            math.fsum(absolute_deltas) / len(absolute_deltas)
                            if absolute_deltas else None
                        ),
                        'rms_difference': (
                            math.sqrt(math.fsum(delta * delta for delta in deltas) / len(deltas))
                            if deltas else None
                        ),
                        'maximum_absolute_difference': max(absolute_deltas) if absolute_deltas else None,
                    })
                key = (role, harmonic, metric)
                figures[key] = plot_sweep_repeatability(
                    rows, role=role, harmonic=harmonic, metric=metric,
                    baseline_source_path=baseline, excitation_path=excitation_path,
                    excitation_x_axis=excitation_x_axis_widget.value,
                )
    return figures

if frequency_rows:
    if len({row.source_path for row in frequency_rows}) > 1:
        frequency_figures = _repeatability_figures(
            frequency_rows, frequency_baseline_widget.value, frequency_excitation_path
        )
    else:
        frequency_figures = plot_six_role_harmonic_sweeps(
            frequency_rows, excitation_path=frequency_excitation_path,
            phase_minimum_amplitude_v=PHASE_MINIMUM_AMPLITUDE_V,
            phase_maximum_standard_deviation_deg=PHASE_MAXIMUM_STANDARD_DEVIATION_DEG,
        )
if excitation_rows:
    if len({row.source_path for row in excitation_rows}) > 1:
        current_voltage_figures = _repeatability_figures(
            excitation_rows, excitation_baseline_widget.value, excitation_excitation_path
        )
    else:
        current_voltage_figures = plot_six_role_harmonic_sweeps(
            excitation_rows, excitation_path=excitation_excitation_path,
            excitation_x_axis=excitation_x_axis_widget.value,
            phase_minimum_amplitude_v=PHASE_MINIMUM_AMPLITUDE_V,
            phase_maximum_standard_deviation_deg=PHASE_MAXIMUM_STANDARD_DEVIATION_DEG,
        )
if combined_rows:
    if len({row.source_path for row in combined_rows}) > 1:
        combined_iv_figures = _repeatability_figures(
            combined_rows, combined_baseline_widget.value, combined_excitation_path
        )
    else:
        combined_iv_figures = {
            (role, harmonic): plot_multi_frequency_iv_curves(
                combined_rows, role=role, harmonic=harmonic, metric='amplitude_v',
                excitation_path=combined_excitation_path,
                excitation_x_axis=excitation_x_axis_widget.value,
            )
            for role in ('xx', 'xy') for harmonic in (1, 2, 3)
            if any(row.role == role and row.harmonic == harmonic for row in combined_rows)
        }

for figures in (frequency_figures, current_voltage_figures, combined_iv_figures):
    for figure in figures.values():
        display(figure)
        plt.close(figure)
if repeatability_summary:
    summary_headers = ('Scan', 'Run', 'Baseline', 'Channel', 'Metric', 'Matched points', 'Mean |Δ|', 'RMS Δ', 'Max |Δ|')
    summary_head = ''.join(f'<th>{html.escape(label)}</th>' for label in summary_headers)
    summary_body = ''.join(
        '<tr>' + ''.join(f'<td>{html.escape(str(value))}</td>' for value in (
            item['scan_type'], Path(item['source_path']).name,
            Path(item['baseline_source_path']).name,
            f"{item['role']} h{item['harmonic']}", item['metric'],
            item['matched_coordinates'], item['mean_absolute_difference'],
            item['rms_difference'], item['maximum_absolute_difference'],
        )) + '</tr>'
        for item in repeatability_summary
    )
    display(widgets.HTML(
        '<b>Paired repeatability summary</b>: differences use only shared requested coordinates; '
        'phase deltas use the shortest signed circular difference.<div style="overflow-x:auto"><table>'
        + '<thead><tr>' + summary_head + '</tr></thead><tbody>' + summary_body + '</tbody></table></div>'
    ))

if excitation_rows:
    for source_path in dict.fromkeys(row.source_path for row in excitation_rows):
        run_rows = tuple(row for row in excitation_rows if row.source_path == source_path)
        try:
            run_excitation_path = excitation_path_from_sweep_files(
                (Path(source_path),), excitation_path_override=EXCITATION_PATH_OVERRIDE,
            )
        except ValueError as error:
            if excitation_x_axis_widget.value != 'sine_output_v_rms':
                raise
            print(f'Skipping current-based fits for {Path(source_path).name}: {error}')
            continue
        run_results = fit_harmonic_scalings(
            run_rows, excitation_path=run_excitation_path, rules=SCALING_RULES,
        )
        harmonic_scaling_results_by_run[source_path] = run_results
        for key, result in run_results.items():
            figure = plot_harmonic_scaling_fit(result, methods=SCALING_PLOT_METHODS)
            figure.axes[0].set_title(figure.axes[0].get_title() + f' · {Path(source_path).name}')
            harmonic_scaling_figures[(source_path, *key)] = figure
    selected_baseline = excitation_baseline_widget.value
    if selected_baseline in harmonic_scaling_results_by_run:
        harmonic_scaling_results = harmonic_scaling_results_by_run[selected_baseline]
    elif harmonic_scaling_results_by_run:
        harmonic_scaling_results = next(iter(harmonic_scaling_results_by_run.values()))
    for figure in harmonic_scaling_figures.values():
        display(figure)
        plt.close(figure)

## Condensed report figure

This independent cell combines any selected Vxx/Vxy harmonic amplitudes in one figure. Optional phase data use a clearly labeled right-hand axis; set `REPORT_PHASE_MODE = 'none'` and `REPORT_PHASE_CHANNELS = ()` for an amplitude-only figure. Each amplitude channel shows only its final selected free-exponent scalar-R model, with the fitted equation and parameters in the right-side legend. Existing analysis figures and fit comparisons are unchanged.

In [ ]:
from attodry_control.report_plotting import (
    condensed_iv_report_manifest,
    plot_condensed_iv_report,
)
from attodry_control.scientific_plotting import export_publication_figure_set

if not harmonic_scaling_results_by_run:
    raise RuntimeError(
        "Load at least one excitation run and run the per-run fit cell first."
    )

# Each report uses one run's fits; fits from separate runs are never pooled.
# None keeps every available channel; otherwise list keys such as (("xx", 1), ("xy", 2)).
REPORT_AMPLITUDE_CHANNELS = None
REPORT_PHASE_MODE = "none"
# REPORT_PHASE_MODE = "right"  # Optional phase on the right axis.
REPORT_PHASE_CHANNELS = ()
REPORT_TITLE = "Selected lock-in I–V summary"
# Leave as None to display only, or set a suffix-free path for PNG/PDF/SVG.
REPORT_OUTPUT_STEM = None
# REPORT_OUTPUT_STEM = OUTPUT_DIRECTORY / "condensed_iv_report"
report_figures = {}
for report_index, (report_source_path, report_results) in enumerate(
    harmonic_scaling_results_by_run.items(), start=1
):
    REPORT_AVAILABLE_CHANNELS = tuple(
        key for key, fit in sorted(report_results.items())
        if fit.scalar_selected_free_model is not None
    )
    report_channels = (
        REPORT_AVAILABLE_CHANNELS
        if REPORT_AMPLITUDE_CHANNELS is None
        else tuple(key for key in REPORT_AMPLITUDE_CHANNELS if key in report_results)
    )
    if not report_channels:
        continue
    report_figure = plot_condensed_iv_report(
        report_results,
        amplitude_channels=report_channels,
        phase_channels=REPORT_PHASE_CHANNELS,
        phase_mode=REPORT_PHASE_MODE,
        title=f"{REPORT_TITLE} · {Path(report_source_path).stem}",
    )
    report_figures[report_source_path] = report_figure
    display(report_figure)
    if REPORT_OUTPUT_STEM is not None:
        report_base = Path(REPORT_OUTPUT_STEM)
        report_output_stem = report_base.with_name(
            f"{report_base.name}_{report_index:02d}_{Path(report_source_path).stem}"
        )
        exported_report_paths = export_publication_figure_set(
            report_figure, report_output_stem
        )
        report_manifest = condensed_iv_report_manifest(
            report_results,
            amplitude_channels=report_channels,
            phase_channels=REPORT_PHASE_CHANNELS,
            phase_mode=REPORT_PHASE_MODE,
            source_paths=(Path(report_source_path),),
        )
        report_manifest_path = report_output_stem.with_name(
            report_output_stem.name + "_manifest.json"
        )
        report_manifest_path.write_text(
            json.dumps(report_manifest, indent=2), encoding="utf-8"
        )
        display((*exported_report_paths, report_manifest_path))
    plt.close(report_figure)
if not report_figures:
    raise RuntimeError("No excitation run has an available scalar report channel.")

## Optional export

No files are written unless `SAVE_OUTPUTS=True`. Exports are placed under the ignored analysis-output directory and include a JSON selection manifest with filters, chosen files, excluded scan points, selected temperature conditions, the editable scaling rules, and each fit result.

In [ ]:
SAVE_OUTPUTS = False
OUTPUT_DIRECTORY = PROJECT_ROOT / 'analysis_output' / 'sr830_commissioning'
if SAVE_OUTPUTS:
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    if frequency_rows:
        export_commissioning_csv(
            frequency_rows, OUTPUT_DIRECTORY / 'frequency_samples.csv'
        )
    if excitation_rows:
        export_commissioning_csv(
            excitation_rows, OUTPUT_DIRECTORY / 'excitation_samples.csv'
        )
    if combined_rows:
        export_commissioning_csv(
            combined_rows, OUTPUT_DIRECTORY / 'frequency_excitation_samples.csv'
        )
    if temperature_excitation_rows:
        export_temperature_excitation_csv(
            temperature_excitation_rows,
            OUTPUT_DIRECTORY / 'temperature_excitation_samples.csv',
        )
    for scan_name, figures in (
        ('frequency', frequency_figures),
        ('current_voltage', current_voltage_figures),
        ('combined_current_voltage', combined_iv_figures),
    ):
        for figure_key, figure in figures.items():
            if len(figure_key) == 2:
                role, harmonic = figure_key
                stem = f'{scan_name}_{role}_h{harmonic}'
            else:
                role, harmonic, metric = figure_key
                stem = f'{scan_name}_{role}_h{harmonic}_{metric}_repeatability'
            export_publication_figure_set(figure, OUTPUT_DIRECTORY / stem)
    for (role, harmonic, metric), figure in temperature_iv_figures.items():
        stem = f'temperature_iv_{role}_h{harmonic}_{metric}'
        export_publication_figure_set(figure, OUTPUT_DIRECTORY / stem)
    for (source_path, role, harmonic), figure in harmonic_scaling_figures.items():
        stem = f'harmonic_scaling_{Path(source_path).stem}_{role}_h{harmonic}'
        export_publication_figure_set(figure, OUTPUT_DIRECTORY / stem)
    selection_manifest = {
        'excitation_plot_x_axis': excitation_x_axis_widget.value,
        'data_directory': str(DATA_DIRECTORY),
        'temperature_data_directory': str(TEMPERATURE_DATA_DIRECTORY),
        'filters': {
            'record_statuses': sorted(RECORD_STATUSES) if RECORD_STATUSES else None,
            'sample_statuses': sorted(SAMPLE_STATUSES),
            'include_rejected': INCLUDE_REJECTED,
        },
        'phase_display': {
            'PHASE_MINIMUM_AMPLITUDE_V': PHASE_MINIMUM_AMPLITUDE_V,
            'PHASE_MAXIMUM_STANDARD_DEVIATION_DEG': PHASE_MAXIMUM_STANDARD_DEVIATION_DEG,
        },
        'harmonic_scaling_rules': asdict(SCALING_RULES),
        'harmonic_scaling_plot_methods': list(SCALING_PLOT_METHODS),
        'harmonic_scaling_results': {
            f'{role}_h{harmonic}': result.as_dict()
            for (role, harmonic), result in harmonic_scaling_results.items()
        },
        'harmonic_scaling_results_by_run': {
            source_path: {
                f'{role}_h{harmonic}': result.as_dict()
                for (role, harmonic), result in run_results.items()
            }
            for source_path, run_results in harmonic_scaling_results_by_run.items()
        },
        'repeatability': {
            'metrics': list(repeatability_metrics_widget.value),
            'frequency_baseline': frequency_baseline_widget.value,
            'excitation_baseline': excitation_baseline_widget.value,
            'combined_baseline': combined_baseline_widget.value,
            'matching_coordinates': {
                'frequency': 'target_frequency_hz',
                'excitation': 'source_v_rms',
                'frequency_excitation': ['target_frequency_hz', 'source_v_rms'],
            },
            'phase_difference': 'shortest circular signed difference in [-180, 180) degrees',
        },
        'repeatability_summary': repeatability_summary,
        'frequency': {
            'files': [str(path) for path in frequency_paths],
            'excluded_target_frequency_hz': sorted(FREQUENCY_EXCLUDED_TARGET_HZ),
            'selected_rows': len(frequency_rows),
        },
        'excitation': {
            'files': [str(path) for path in excitation_paths],
            'excluded_source_v_rms': sorted(EXCITATION_EXCLUDED_SOURCE_V_RMS),
            'selected_rows': len(excitation_rows),
        },
        'frequency_excitation': {
            'files': [str(path) for path in combined_paths],
            'excluded_frequencies_hz': sorted(COMBINED_EXCLUDED_FREQUENCIES_HZ),
            'excluded_excitations_v_rms': sorted(COMBINED_EXCLUDED_EXCITATIONS_V_RMS),
            'selected_rows': len(combined_rows),
        },
        'temperature_excitation': {
            'files': [str(path) for path in temperature_excitation_paths],
            'sample_statuses': sorted(temperature_sample_status_widget.value),
            'selected_condition_keys': sorted(temperature_condition_widget.value),
            'current_minimum_a_rms': temperature_current_minimum_a_rms,
            'current_maximum_a_rms': temperature_current_maximum_a_rms,
            'selected_rows': len(temperature_excitation_rows),
            'phase_statistics': 'circular mean/std; unwrapped along current for display',
        },
    }
    (OUTPUT_DIRECTORY / 'selection_manifest.json').write_text(
        json.dumps(selection_manifest, indent=2), encoding='utf-8'
    )
    display(OUTPUT_DIRECTORY)